# 02 — Pipeline Frank VCG: MI so với Healthy Control

Mục tiêu là nhìn rõ đường đi của dữ liệu:

```text
PTB record → Frank X/Y/Z → preprocessing → R peaks → beats
→ representative beat → QRS/T delineation → QRS/T loops → PCA
→ 12 features → thống kê → patient-level split → ML → MI / HC
```

Notebook ưu tiên khả năng kiểm tra bằng mắt. QRS/T delineation ở đây là một implementation tham khảo dựa trên năng lượng/chuyển động của vector, **chưa phải ground truth lâm sàng** và phải được thẩm định trước khi dùng làm kết quả nghiên cứu.


## Chuẩn bị dữ liệu trên máy này
Chạy cell dưới một lần mỗi phiên. Dữ liệu PTB được tải từ GitHub Releases nếu còn thiếu.
Notebook 06–09 tự tạo các đầu vào phụ thuộc còn thiếu; lần đầu có thể chạy lâu.
Hướng dẫn và dung lượng: xem `README.md` và `docs/REPRODUCIBILITY.md`.

In [ ]:
from project_setup import prepare_notebook
prepare_notebook(2)

## A. Thư viện và cấu hình

Nếu thiếu thư viện, bỏ dấu `#` ở dòng `%pip`, chạy một lần rồi restart kernel. ML chỉ nhận đúng 12 VCG features; metadata không được đưa vào input.


In [ ]:
# %pip install numpy pandas matplotlib scipy scikit-learn wfdb

from pathlib import Path
from collections import Counter
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter, butter, sosfiltfilt, find_peaks
from scipy.stats import mannwhitneyu
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, ConfusionMatrixDisplay)
import wfdb

pd.set_option("display.max_columns", 100)
plt.rcParams["figure.figsize"] = (12, 4)

DATA_ROOT = Path("content/ptbdb/physionet.org/files/ptbdb/1.0.0")
PRE_R_MS = 300
POST_R_MS = 500
SG_POLYORDER = 2
SG_WINDOW_SAMPLES = 1201
ANALYSIS_SECONDS = 30       # None = dùng toàn recording; 30 giây giúp chạy minh họa nhanh hơn
RUN_FULL_DATASET = True     # False = chỉ smoke-test 1 MI + 1 HC
TEST_SIZE = 0.25
RANDOM_STATE = 42

if not DATA_ROOT.exists():
    raise FileNotFoundError(f"Không tìm thấy {DATA_ROOT.resolve()}; hãy sửa DATA_ROOT.")
print("DATA_ROOT:", DATA_ROOT.resolve())


## A1. Đọc metadata và chọn MI/HC

Diagnosis được lấy đúng từ `Reason for admission`. Mapping là exact-match sau chuẩn hóa; diagnosis khác bị loại khỏi pipeline. Không dùng file `CONTROLS` để thay cho diagnosis trong header, nhờ vậy cùng một logic hoạt động với cả MI và HC.


In [ ]:
MI_DIAGNOSES = {"myocardial infarction"}
HC_DIAGNOSES = {"healthy control", "healthy controls"}

FEATURE_NAMES = [
    "QRS_Arc", "QRS_Area", "QRS_MaxVec", "QRS_MaxCentroidDistance",
    "QRS_MaxVel", "QRS_MeanVel", "QRS_StdVel",
    "T_MaxVec", "T_Area", "T_MaxVel", "T_MeanVel", "T_StdVel",
]


def parse_header_metadata(header_text):
    """Input: raw header. Output: dictionary field-value, bảo toàn các field lặp."""
    result, counts = {}, Counter()
    for raw_line in header_text.splitlines():
        content = raw_line.strip()
        if not content.startswith("#") or ":" not in content:
            continue
        field, value = (part.strip() for part in content[1:].split(":", 1))
        if not field:
            continue
        counts[field] += 1
        key = field if counts[field] == 1 else f"{field}__{counts[field]}"
        result[key] = value or np.nan
    return result


def map_diagnosis_group(value):
    """Input: diagnosis raw. Output: MI, HC hoặc OTHER bằng exact mapping."""
    if pd.isna(value):
        return "OTHER"
    normalized = " ".join(str(value).strip().lower().split())
    if normalized in MI_DIAGNOSES:
        return "MI"
    if normalized in HC_DIAGNOSES:
        return "HC"
    return "OTHER"


def patient_id_from_path(record_base):
    """Input: record path. Output: patient_id lấy từ thành phần patientNNN hoặc folder cha."""
    for part in reversed(Path(record_base).parts):
        if re.fullmatch(r"patient\d+", part, flags=re.IGNORECASE):
            return part.lower()
    return Path(record_base).parent.name


def scan_ptb_records(data_root):
    """Input: DATA_ROOT. Output: DataFrame index gồm path, ID, diagnosis và label."""
    rows = []
    for hea in sorted(Path(data_root).rglob("*.hea")):
        base = hea.with_suffix("")
        meta = parse_header_metadata(hea.read_text(encoding="latin-1"))
        diagnosis = meta.get("Reason for admission", np.nan)
        rows.append({
            "record_path": base,
            "record_id": base.relative_to(data_root).as_posix(),
            "patient_id": patient_id_from_path(base),
            "diagnosis": diagnosis,
            "label": map_diagnosis_group(diagnosis),
        })
    return pd.DataFrame(rows)


df_index_all = scan_ptb_records(DATA_ROOT)
print("Diagnosis thực tế trước mapping:")
display(df_index_all.groupby("diagnosis", dropna=False).agg(
    n_records=("record_id", "size"), n_patients=("patient_id", "nunique")
).sort_values("n_records", ascending=False))

df_index = df_index_all[df_index_all["label"].isin(["MI", "HC"])].reset_index(drop=True)
display(df_index.groupby("label").agg(
    n_patients=("patient_id", "nunique"), n_records=("record_id", "size")
).reindex(["MI", "HC"]))


## B. Chọn 1 MI và 1 HC để demo

Chọn record đầu tiên sau khi sort đường dẫn: xác định và tái lập được, không random. Các hình demo giúp bắt lỗi trước khi chạy toàn dataset.


In [ ]:
def select_demo_record(index_df, label):
    """Input: bảng index và label. Output: record đầu tiên theo record_id, không random."""
    candidates = index_df[index_df["label"] == label].sort_values("record_id")
    if candidates.empty:
        raise ValueError(f"Không có record {label}")
    return candidates.iloc[0]


demo_mi = select_demo_record(df_index, "MI")
demo_hc = select_demo_record(df_index, "HC")
print("MI demo:", demo_mi["record_id"], "—", demo_mi["patient_id"])
print("HC demo:", demo_hc["record_id"], "—", demo_hc["patient_id"])


## B1. Load Frank X/Y/Z

Function tìm channel theo tên/alias, không lấy “ba cột cuối”. Trong PTB hiện tại tên là `vx`, `vy`, `vz`.


In [ ]:
def get_xyz_channels(channel_names):
    """Input: danh sách channel. Output: mapping trục X/Y/Z sang tên channel thật."""
    aliases = {"X": {"x", "vx", "frankx", "frank_x"},
               "Y": {"y", "vy", "franky", "frank_y"},
               "Z": {"z", "vz", "frankz", "frank_z"}}
    normalized = {name: re.sub(r"[^a-z0-9_]", "", str(name).lower()) for name in channel_names}
    result = {}
    for axis, accepted in aliases.items():
        hits = [original for original, norm in normalized.items() if norm in accepted]
        if len(hits) != 1:
            raise ValueError(f"Không xác định duy nhất {axis}; hits={hits}, channels={channel_names}")
        result[axis] = hits[0]
    return result


def choose_detection_channel(channel_names):
    """Input: danh sách channel. Output: tên ECG lead dùng để phát hiện R peak."""
    lower_to_original = {str(name).lower(): name for name in channel_names}
    for preferred in ["ii", "i", "v2"]:
        if preferred in lower_to_original:
            return lower_to_original[preferred]
    xyz_names = set(get_xyz_channels(channel_names).values())
    candidates = [name for name in channel_names if name not in xyz_names]
    if not candidates:
        raise ValueError("Không có ECG lead để R-peak detection")
    return candidates[0]


def load_ptb_record(record_path, max_seconds=None):
    """Input: record path và giới hạn giây. Output: XYZ, ECG detection lead, fs và metadata kênh."""
    header = wfdb.rdheader(str(record_path))
    xyz_names = get_xyz_channels(header.sig_name)
    r_name = choose_detection_channel(header.sig_name)
    selected = [xyz_names[axis] for axis in "XYZ"] + [r_name]
    sampto = None if max_seconds is None else min(header.sig_len, int(max_seconds * header.fs))
    record = wfdb.rdrecord(str(record_path), channel_names=selected, sampto=sampto)
    columns = {name: record.p_signal[:, i].astype(float) for i, name in enumerate(record.sig_name)}
    xyz = np.column_stack([columns[xyz_names[axis]] for axis in "XYZ"])
    return {"xyz": xyz, "r_signal": columns[r_name], "r_channel": r_name,
            "fs": float(record.fs), "xyz_names": xyz_names, "units": record.units}


demo_raw = load_ptb_record(demo_mi["record_path"], max_seconds=ANALYSIS_SECONDS)
seconds_to_plot = min(8, len(demo_raw["xyz"]) / demo_raw["fs"])
n_plot = int(seconds_to_plot * demo_raw["fs"])
t = np.arange(n_plot) / demo_raw["fs"]

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
for axis_i, (axis_name, ax) in enumerate(zip("XYZ", axes)):
    ax.plot(t, demo_raw["xyz"][:n_plot, axis_i], linewidth=0.8)
    ax.set_ylabel(axis_name)
    ax.grid(alpha=0.2)
axes[-1].set_xlabel("Thời gian (giây)")
fig.suptitle(f"Frank X/Y/Z gốc — MI demo: {demo_mi['record_id']}")
plt.tight_layout()
plt.show()


## C. Preprocessing bằng Savitzky–Golay

Theo cấu hình nghiên cứu: polynomial order = 2, window mong muốn = 1201 sample. Function luôn bảo đảm window lẻ, lớn hơn polynomial order và nhỏ hơn/không vượt chiều dài tín hiệu. Đây là phép làm mượt; hình raw-vs-filtered là bước QC bắt buộc. Tham số 1201 khá lớn ở 1000 Hz, vì vậy cần đối chiếu paper/code gốc xem mục tiêu thực sự là làm mượt hay ước lượng baseline.


In [ ]:
def safe_savgol_window(n_samples, desired_window, polyorder):
    """Input: chiều dài, window mong muốn, polyorder. Output: window lẻ hợp lệ."""
    if n_samples <= polyorder + 1:
        raise ValueError("Tín hiệu quá ngắn cho Savitzky–Golay")
    window = min(int(desired_window), n_samples if n_samples % 2 == 1 else n_samples - 1)
    if window % 2 == 0:
        window -= 1
    minimum = polyorder + 2
    if minimum % 2 == 0:
        minimum += 1
    if window < minimum:
        window = minimum
    if window > n_samples:
        raise ValueError("Không tạo được Savitzky–Golay window hợp lệ")
    return window


def preprocess_xyz(xyz, window_samples=1201, polyorder=2):
    """Input: mảng (n,3). Output: XYZ đã Savitzky-Golay và window thực dùng."""
    xyz = np.asarray(xyz, dtype=float)
    if xyz.ndim != 2 or xyz.shape[1] != 3:
        raise ValueError(f"xyz phải có shape (n_samples, 3), nhận {xyz.shape}")
    window = safe_savgol_window(len(xyz), window_samples, polyorder)
    filtered = np.column_stack([
        savgol_filter(xyz[:, i], window_length=window, polyorder=polyorder, mode="interp")
        for i in range(3)
    ])
    return filtered, window


demo_filtered, used_window = preprocess_xyz(demo_raw["xyz"], SG_WINDOW_SAMPLES, SG_POLYORDER)
print("Savitzky–Golay window thực dùng:", used_window, "samples")
start = int(2 * demo_raw["fs"])
stop = min(len(demo_filtered), start + int(2 * demo_raw["fs"]))
t_small = np.arange(start, stop) / demo_raw["fs"]

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
for i, (name, ax) in enumerate(zip("XYZ", axes)):
    ax.plot(t_small, demo_raw["xyz"][start:stop, i], alpha=0.45, label="raw")
    ax.plot(t_small, demo_filtered[start:stop, i], linewidth=1.5, label="filtered")
    ax.set_ylabel(name)
    ax.legend()
axes[-1].set_xlabel("Thời gian (giây)")
fig.suptitle("Raw vs Savitzky–Golay filtered")
plt.tight_layout()
plt.show()


## D. R-peak detection

R peak là mốc trung tâm của mỗi nhịp. Ta dùng lead II (nếu có), band-pass 5–20 Hz để nhấn QRS, `find_peaks` với khoảng cách tối thiểu 300 ms, rồi tinh chỉnh peak trên tín hiệu gốc trong ±80 ms. Đây là detector minh họa; luôn xem đồ thị QC.


In [ ]:
def detect_r_peaks(ecg_signal, fs):
    """Input: ECG 1D và fs. Output: index R peaks và tín hiệu band-pass dùng cho QC."""
    ecg = np.asarray(ecg_signal, dtype=float)
    if len(ecg) < int(2 * fs):
        raise ValueError("Tín hiệu quá ngắn để phát hiện R peak")
    nyquist = fs / 2
    high = min(20.0, 0.95 * nyquist)
    low = min(5.0, 0.5 * high)
    sos = butter(3, [low / nyquist, high / nyquist], btype="bandpass", output="sos")
    bandpassed = sosfiltfilt(sos, np.nan_to_num(ecg, nan=np.nanmedian(ecg)))
    envelope = np.abs(bandpassed)
    threshold = np.median(envelope) + 2.5 * np.median(np.abs(envelope - np.median(envelope)))
    peaks, _ = find_peaks(envelope, distance=max(1, int(0.30 * fs)), prominence=max(threshold, 1e-12))

    radius = max(1, int(0.08 * fs))
    baseline = np.median(ecg)
    refined = []
    for peak in peaks:
        lo, hi = max(0, peak - radius), min(len(ecg), peak + radius + 1)
        refined.append(lo + int(np.argmax(np.abs(ecg[lo:hi] - baseline))))
    refined = np.array(sorted(set(refined)), dtype=int)
    if len(refined) < 3:
        raise ValueError(f"Chỉ tìm thấy {len(refined)} R peaks")
    return refined, bandpassed


demo_r_peaks, demo_r_bandpassed = detect_r_peaks(demo_raw["r_signal"], demo_raw["fs"])
print(f"Tìm thấy {len(demo_r_peaks)} R peaks trên {demo_raw['r_channel']}")

n_qc = min(len(demo_raw["r_signal"]), int(10 * demo_raw["fs"]))
t_qc = np.arange(n_qc) / demo_raw["fs"]
visible_peaks = demo_r_peaks[demo_r_peaks < n_qc]
plt.figure(figsize=(13, 4))
plt.plot(t_qc, demo_raw["r_signal"][:n_qc], linewidth=0.8, label=demo_raw["r_channel"])
plt.scatter(visible_peaks / demo_raw["fs"], demo_raw["r_signal"][visible_peaks], color="red", zorder=3)
for number, peak in enumerate(visible_peaks, 1):
    plt.annotate(f"R{number}", (peak / demo_raw["fs"], demo_raw["r_signal"][peak]), fontsize=8)
plt.xlabel("Thời gian (giây)")
plt.title("QC R-peak detection")
plt.legend()
plt.show()


## E. Beat extraction

Một `beat` là một cửa sổ quanh R peak. Mặc định lấy 300 ms trước R và 500 ms sau R. Peak quá gần biên bị loại; mọi beat còn lại có cùng số sample.


In [ ]:
def extract_beats(xyz, r_peaks, fs, pre_r_ms=300, post_r_ms=500):
    """Input: XYZ, R peaks, fs và cửa sổ ms. Output: beats, peaks được giữ và R index."""
    pre = int(round(pre_r_ms * fs / 1000))
    post = int(round(post_r_ms * fs / 1000))
    beats, kept_peaks = [], []
    for peak in np.asarray(r_peaks, dtype=int):
        start, stop = peak - pre, peak + post + 1
        if start >= 0 and stop <= len(xyz):
            beats.append(xyz[start:stop])
            kept_peaks.append(peak)
    if not beats:
        raise ValueError("Không có beat hợp lệ sau khi loại biên")
    return np.stack(beats), np.asarray(kept_peaks), pre


demo_beats, demo_kept_peaks, demo_r_index = extract_beats(
    demo_filtered, demo_r_peaks, demo_raw["fs"], PRE_R_MS, POST_R_MS
)
print("beats shape:", demo_beats.shape, "= (số beat, số thời điểm, X/Y/Z)")
beat_time_ms = (np.arange(demo_beats.shape[1]) - demo_r_index) / demo_raw["fs"] * 1000

fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
for axis_i, ax in enumerate(axes):
    for beat in demo_beats:
        ax.plot(beat_time_ms, beat[:, axis_i], alpha=0.20, linewidth=0.7)
    ax.axvline(0, color="black", linestyle="--", linewidth=1)
    ax.set_ylabel("XYZ"[axis_i])
axes[-1].set_xlabel("Thời gian so với R (ms)")
fig.suptitle("Các beat chồng lên nhau")
plt.tight_layout()
plt.show()


## F. Representative beat

Representative beat là median theo từng thời điểm của tất cả beat trong một recording. Median giảm ảnh hưởng của nhịp nhiễu. Sau đó trừ median của 100 ms đầu để đưa baseline vector gần gốc tọa độ; đây chỉ là căn chỉnh, không tạo feature mới.


In [ ]:
def build_representative_beat(beats, fs, baseline_ms=100):
    """Input: tensor beats và fs. Output: median beat đã căn baseline."""
    representative = np.median(np.asarray(beats), axis=0)
    baseline_n = max(1, min(len(representative), int(baseline_ms * fs / 1000)))
    representative = representative - np.median(representative[:baseline_n], axis=0)
    return representative


demo_rep = build_representative_beat(demo_beats, demo_raw["fs"])
fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
for axis_i, ax in enumerate(axes):
    for beat in demo_beats:
        ax.plot(beat_time_ms, beat[:, axis_i], color="gray", alpha=0.12, linewidth=0.6)
    ax.plot(beat_time_ms, demo_rep[:, axis_i], color="crimson", linewidth=2, label="representative")
    ax.set_ylabel("XYZ"[axis_i])
    ax.legend()
axes[-1].set_xlabel("Thời gian so với R (ms)")
fig.suptitle("Tất cả beat (mờ) và representative beat (đậm)")
plt.tight_layout()
plt.show()


## G. Vector magnitude


In [ ]:
def compute_vector_magnitude(xyz):
    """Input: XYZ (n,3). Output: M(t), norm Euclid tại từng thời điểm."""
    return np.linalg.norm(np.asarray(xyz), axis=1)


demo_magnitude = compute_vector_magnitude(demo_rep)
plt.figure(figsize=(11, 4))
plt.plot(beat_time_ms, demo_magnitude)
plt.axvline(0, color="black", linestyle="--", linewidth=1)
plt.xlabel("Thời gian so với R (ms)")
plt.ylabel("M(t)")
plt.title("Độ lớn tổng hợp của vector VCG")
plt.show()


## H. QRS/T delineation — implementation tham khảo

`QRS_on/off` và `T_on/off` là ranh giới các sóng. Heuristic dưới đây:

1. Tính tốc độ thay đổi của vector XYZ và làm mượt nhẹ.
2. Quanh R, tìm vùng hoạt động QRS vượt ngưỡng so với baseline.
3. Sau QRS, dùng độ lớn vector để tìm T peak và ngưỡng 10% biên độ T.
4. Ép thứ tự hợp lệ `QRS_on < QRS_off < T_on < T_off`.

`TODO` trong code là cảnh báo nghiên cứu thật phải validate với paper/code tham chiếu hoặc annotation chuyên gia.


In [ ]:
def _nearest_below_run(values, start, direction, threshold, run_length, limit):
    """Input: trace, điểm bắt đầu, hướng và ngưỡng. Output: index biên có đoạn dưới ngưỡng."""
    if direction < 0:
        indices = range(start, limit - 1, -1)
    else:
        indices = range(start, limit + 1)
    count = 0
    for idx in indices:
        count = count + 1 if values[idx] <= threshold else 0
        if count >= run_length:
            return idx + run_length - 1 if direction < 0 else idx - run_length + 1
    return limit


def delineate_qrs_t(representative_xyz, fs, r_index):
    """Input: representative XYZ, fs, R index. Output: bốn boundary và trace QC."""
    # TODO: validate delineation method against reference paper/code
    xyz = np.asarray(representative_xyz, dtype=float)
    n = len(xyz)
    speed = np.linalg.norm(np.gradient(xyz, 1 / fs, axis=0), axis=1)
    smooth_window = safe_savgol_window(n, max(5, int(0.021 * fs) | 1), 2)
    activity = savgol_filter(speed, smooth_window, 2)

    q_lo = max(0, r_index - int(0.15 * fs))
    q_hi = min(n - 1, r_index + int(0.18 * fs))
    baseline_zone = np.r_[activity[:max(1, q_lo)], activity[min(n, r_index + int(0.45 * fs)):]]
    baseline = float(np.median(baseline_zone)) if baseline_zone.size else float(np.median(activity))
    q_peak = float(np.max(activity[q_lo:q_hi + 1]))
    q_threshold = baseline + 0.12 * max(q_peak - baseline, 0)
    run = max(2, int(0.012 * fs))
    qrs_on = _nearest_below_run(activity, r_index, -1, q_threshold, run, q_lo)
    qrs_off = _nearest_below_run(activity, r_index, +1, q_threshold, run, q_hi)

    magnitude = compute_vector_magnitude(xyz)
    t_search_lo = min(n - 2, qrs_off + int(0.04 * fs))
    t_search_hi = min(n - 1, r_index + int(0.45 * fs))
    if t_search_hi <= t_search_lo + 2:
        raise ValueError("Không đủ vùng tìm T sau QRS")
    t_peak = t_search_lo + int(np.argmax(magnitude[t_search_lo:t_search_hi + 1]))
    t_baseline = float(np.median(np.r_[magnitude[:max(1, qrs_on)], magnitude[max(t_search_hi, 0):]]))
    t_threshold = t_baseline + 0.10 * max(float(magnitude[t_peak]) - t_baseline, 0)
    t_run = max(2, int(0.015 * fs))
    t_on = _nearest_below_run(magnitude, t_peak, -1, t_threshold, t_run, t_search_lo)
    t_off = _nearest_below_run(magnitude, t_peak, +1, t_threshold, t_run, t_search_hi)

    qrs_on = int(np.clip(qrs_on, 0, r_index - 1))
    qrs_off = int(np.clip(qrs_off, r_index + 1, t_search_lo - 1))
    t_on = int(np.clip(t_on, qrs_off + 1, t_peak))
    t_off = int(np.clip(t_off, t_peak + 1, n - 1))
    if not (qrs_on < qrs_off < t_on < t_off):
        raise ValueError(f"Boundary không hợp lệ: {qrs_on, qrs_off, t_on, t_off}")
    return {"qrs_on": qrs_on, "qrs_off": qrs_off, "t_on": t_on, "t_off": t_off,
            "activity": activity, "magnitude": magnitude}


demo_bounds = delineate_qrs_t(demo_rep, demo_raw["fs"], demo_r_index)
plt.figure(figsize=(12, 4))
plt.plot(beat_time_ms, demo_bounds["magnitude"], label="M(t)")
colors = {"qrs_on": "green", "qrs_off": "green", "t_on": "purple", "t_off": "purple"}
for name in ["qrs_on", "qrs_off", "t_on", "t_off"]:
    idx = demo_bounds[name]
    plt.axvline(beat_time_ms[idx], color=colors[name], linestyle="--", label=name)
plt.xlabel("Thời gian so với R (ms)")
plt.title("QC bằng mắt: ranh giới QRS và T trên representative beat")
plt.legend(ncol=4)
plt.show()
print({key: f"{beat_time_ms[value]:.1f} ms" for key, value in demo_bounds.items() if key in {"qrs_on", "qrs_off", "t_on", "t_off"}})


## I. Tạo QRS loop và T loop trong không gian 3D


In [ ]:
def build_qrs_loop(representative_xyz, boundaries):
    """Input: representative XYZ và boundary. Output: QRS loop có shape (n_qrs,3)."""
    return np.asarray(representative_xyz)[boundaries["qrs_on"]:boundaries["qrs_off"] + 1]


def build_t_loop(representative_xyz, boundaries):
    """Input: representative XYZ và boundary. Output: T loop có shape (n_t,3)."""
    return np.asarray(representative_xyz)[boundaries["t_on"]:boundaries["t_off"] + 1]


demo_qrs_loop = build_qrs_loop(demo_rep, demo_bounds)
demo_t_loop = build_t_loop(demo_rep, demo_bounds)
print("QRS loop shape:", demo_qrs_loop.shape)
print("T loop shape:", demo_t_loop.shape)

for name, loop, color in [("QRS", demo_qrs_loop, "crimson"), ("T", demo_t_loop, "purple")]:
    fig = plt.figure(figsize=(7, 6))
    ax = fig.add_subplot(111, projection="3d")
    ax.plot(loop[:, 0], loop[:, 1], loop[:, 2], color=color)
    ax.scatter(loop[0, 0], loop[0, 1], loop[0, 2], color="green", label="start")
    ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")
    ax.set_title(f"{name} loop 3D")
    ax.legend()
    plt.show()


## J. PCA cho từng loop

PCA tìm mặt phẳng 2D giữ nhiều biến thiên nhất của **mỗi loop**, để tính Arc và Area. Đây không phải bước giảm 12 features trước ML.


In [ ]:
def project_loop_pca(loop_3d):
    """Input: loop 3D. Output: loop 2D, PCA model và explained-variance ratio."""
    loop = np.asarray(loop_3d, dtype=float)
    if loop.ndim != 2 or loop.shape[1] != 3 or len(loop) < 3:
        raise ValueError(f"Loop không hợp lệ: {loop.shape}")
    model = PCA(n_components=2)
    projected = model.fit_transform(loop)
    return projected, model, model.explained_variance_ratio_


demo_qrs_2d, demo_qrs_pca, qrs_ratio = project_loop_pca(demo_qrs_loop)
demo_t_2d, demo_t_pca, t_ratio = project_loop_pca(demo_t_loop)
for name, points, ratio, color in [
    ("QRS", demo_qrs_2d, qrs_ratio, "crimson"), ("T", demo_t_2d, t_ratio, "purple")
]:
    plt.figure(figsize=(6, 6))
    plt.plot(points[:, 0], points[:, 1], color=color)
    plt.scatter(points[0, 0], points[0, 1], color="green", label="start")
    plt.axis("equal")
    plt.xlabel("PC1"); plt.ylabel("PC2")
    plt.title(f"{name} loop PCA 2D — variance={ratio.sum():.3f}")
    plt.legend(); plt.show()


## K. Tính đúng 12 features

- Arc: tổng độ dài các đoạn liên tiếp trên PCA 2D.
- Area: công thức shoelace, tự đóng điểm cuối về điểm đầu.
- MaxVec: độ lớn vector 3D lớn nhất.
- MaxCentroidDistance: xa centroid nhất (chỉ QRS).
- Velocity: đạo hàm XYZ theo giây; lấy max/mean/std của speed.


In [ ]:
def compute_arc(loop_2d):
    """Input: loop 2D có thứ tự thời gian. Output: tổng độ dài các đoạn liên tiếp."""
    return float(np.linalg.norm(np.diff(np.asarray(loop_2d), axis=0), axis=1).sum())


def compute_area(loop_2d):
    """Input: loop 2D. Output: diện tích polygon theo shoelace formula."""
    points = np.asarray(loop_2d, dtype=float)
    u, v = points[:, 0], points[:, 1]
    return float(0.5 * abs(np.dot(u, np.roll(v, -1)) - np.dot(v, np.roll(u, -1))))


def compute_velocity_features(loop_3d, fs):
    """Input: loop 3D và fs. Output: max, mean, std của speed."""
    velocity = np.gradient(np.asarray(loop_3d, dtype=float), 1 / fs, axis=0)
    speed = np.linalg.norm(velocity, axis=1)
    return {"max": float(np.max(speed)), "mean": float(np.mean(speed)), "std": float(np.std(speed, ddof=0))}


def extract_vcg_features(qrs_loop, t_loop, fs):
    """Input: QRS loop, T loop, fs. Output: dictionary đúng 12 VCG features."""
    qrs_2d, _, _ = project_loop_pca(qrs_loop)
    t_2d, _, _ = project_loop_pca(t_loop)
    qrs_vel = compute_velocity_features(qrs_loop, fs)
    t_vel = compute_velocity_features(t_loop, fs)
    centroid = np.mean(qrs_loop, axis=0)
    features = {
        "QRS_Arc": compute_arc(qrs_2d),
        "QRS_Area": compute_area(qrs_2d),
        "QRS_MaxVec": float(np.linalg.norm(qrs_loop, axis=1).max()),
        "QRS_MaxCentroidDistance": float(np.linalg.norm(qrs_loop - centroid, axis=1).max()),
        "QRS_MaxVel": qrs_vel["max"],
        "QRS_MeanVel": qrs_vel["mean"],
        "QRS_StdVel": qrs_vel["std"],
        "T_MaxVec": float(np.linalg.norm(t_loop, axis=1).max()),
        "T_Area": compute_area(t_2d),
        "T_MaxVel": t_vel["max"],
        "T_MeanVel": t_vel["mean"],
        "T_StdVel": t_vel["std"],
    }
    assert list(features) == FEATURE_NAMES
    return features


demo_features = extract_vcg_features(demo_qrs_loop, demo_t_loop, demo_raw["fs"])
display(pd.DataFrame([demo_features]))


## L. Function cấp cao và demo MI vs HC

`process_record` giữ flow ở một nơi nhưng gọi các function nhỏ để dễ debug. Lỗi không bị bỏ qua; record lỗi sẽ đi vào `df_failures` ở phần toàn dataset.


In [ ]:
def process_record(record_path, metadata, max_seconds=30):
    """Input: record path, metadata, giới hạn giây. Output: feature row và QC row."""
    loaded = load_ptb_record(record_path, max_seconds=max_seconds)
    filtered, used_window = preprocess_xyz(loaded["xyz"], SG_WINDOW_SAMPLES, SG_POLYORDER)
    r_peaks, _ = detect_r_peaks(loaded["r_signal"], loaded["fs"])
    beats, kept_peaks, r_index = extract_beats(filtered, r_peaks, loaded["fs"], PRE_R_MS, POST_R_MS)
    representative = build_representative_beat(beats, loaded["fs"])
    boundaries = delineate_qrs_t(representative, loaded["fs"], r_index)
    qrs_loop = build_qrs_loop(representative, boundaries)
    t_loop = build_t_loop(representative, boundaries)
    features = extract_vcg_features(qrs_loop, t_loop, loaded["fs"])
    return {
        "patient_id": metadata["patient_id"],
        "record_id": metadata["record_id"],
        **features,
        "label": metadata["label"],
    }, {"n_r_peaks": len(r_peaks), "n_beats": len(kept_peaks), "sg_window": used_window,
        **{key: boundaries[key] for key in ["qrs_on", "qrs_off", "t_on", "t_off"]}}


demo_results, demo_qc = {}, {}
for demo_row in [demo_mi, demo_hc]:
    result, qc = process_record(demo_row["record_path"], demo_row, max_seconds=ANALYSIS_SECONDS)
    demo_results[demo_row["label"]] = result
    demo_qc[demo_row["label"]] = qc

comparison = pd.DataFrame({label: [result[name] for name in FEATURE_NAMES]
                           for label, result in demo_results.items()}, index=FEATURE_NAMES)
comparison.index.name = "feature"
display(comparison)

# Chuẩn hóa theo max của hai demo chỉ để các feature khác thang đo cùng nhìn được trên bar chart.
plot_comparison = comparison.div(comparison.abs().max(axis=1).replace(0, 1), axis=0)
plot_comparison.plot.bar(figsize=(13, 5), title="Demo định tính MI vs HC (chuẩn hóa theo từng feature)")
plt.ylabel("Giá trị tương đối")
plt.tight_layout(); plt.show()
print("Lưu ý: hai record chỉ là demo định tính; không đủ để kết luận feature phân biệt MI/HC.")


## M–N. Chạy toàn dataset và tạo `df_features`

Mặc định xử lý mọi MI/HC record, nhưng chỉ dùng 30 giây đầu mỗi recording để notebook thực hành chạy trong thời gian hợp lý. Đặt `ANALYSIS_SECONDS=None` nếu nghiên cứu yêu cầu toàn bộ thời lượng. Đặt `RUN_FULL_DATASET=False` để smoke-test nhanh bằng hai record demo.

Mọi exception được lưu với `record_id` và lý do; không silently ignore. Sau khi chạy, cần xem tỷ lệ failure và QC trước khi phân tích.


In [ ]:
rows_to_process = df_index if RUN_FULL_DATASET else pd.DataFrame([demo_mi, demo_hc])
feature_rows, failure_rows, qc_rows = [], [], []

for position, (_, row) in enumerate(rows_to_process.iterrows(), start=1):
    try:
        result, qc = process_record(row["record_path"], row, max_seconds=ANALYSIS_SECONDS)
        feature_rows.append(result)
        qc_rows.append({"record_id": row["record_id"], **qc})
    except Exception as exc:
        failure_rows.append({"record_id": row["record_id"], "patient_id": row["patient_id"],
                             "label": row["label"], "error": f"{type(exc).__name__}: {exc}"})
    if position % 50 == 0 or position == len(rows_to_process):
        print(f"Đã xử lý {position}/{len(rows_to_process)} — thành công={len(feature_rows)}, lỗi={len(failure_rows)}")

ordered_columns = ["patient_id", "record_id", *FEATURE_NAMES, "label"]
df_features = pd.DataFrame(feature_rows).reindex(columns=ordered_columns)
df_failures = pd.DataFrame(failure_rows)
df_qc = pd.DataFrame(qc_rows)

display(df_features.head())
print("df_features.shape:", df_features.shape)
display(df_features.isna().sum().to_frame("n_missing"))
print(f"Failures: {len(df_failures)}/{len(rows_to_process)}")
if not df_failures.empty:
    display(df_failures)

df_features.to_csv("vcg_12_features.csv", index=False)
df_failures.to_csv("vcg_processing_failures.csv", index=False)
df_qc.to_csv("vcg_processing_qc.csv", index=False)
print("Đã lưu feature, failure và QC CSV.")


## O. Phân tích từng feature

Thống kê dưới đây ở mức **recording**. Vì một bệnh nhân có thể có nhiều recording, các hàng không hoàn toàn độc lập; kết quả khám phá không thay thế mô hình thống kê repeated-measures/patient-level.


In [ ]:
def iqr(series):
    """Input: pandas Series. Output: Q3 trừ Q1."""
    return float(series.quantile(0.75) - series.quantile(0.25))


summary_rows = []
for feature in FEATURE_NAMES:
    for label in ["MI", "HC"]:
        values = df_features.loc[df_features["label"] == label, feature].dropna()
        summary_rows.append({"feature": feature, "label": label, "mean": values.mean(),
                             "median": values.median(), "std": values.std(), "IQR": iqr(values)})
df_feature_summary = pd.DataFrame(summary_rows)
display(df_feature_summary)

for feature in FEATURE_NAMES:
    groups = [df_features.loc[df_features["label"] == label, feature].dropna() for label in ["MI", "HC"]]
    if all(len(values) for values in groups):
        plt.figure(figsize=(6, 4))
        plt.boxplot(groups, tick_labels=["MI", "HC"], showfliers=False)
        plt.ylabel(feature)
        plt.title(f"{feature}: MI vs HC")
        plt.show()


## P. Mann–Whitney U và Benjamini–Hochberg

Mann–Whitney so sánh phân bố hai nhóm mà không giả định chuẩn. `p < 0.05` chỉ là cờ khám phá. Vì kiểm định 12 feature, thêm Benjamini–Hochberg (BH) để kiểm soát false discovery rate. Repeated recordings vẫn là hạn chế quan trọng.


In [ ]:
def benjamini_hochberg(p_values):
    """Input: dãy p-value. Output: BH-adjusted p-value theo thứ tự ban đầu."""
    p = np.asarray(p_values, dtype=float)
    order = np.argsort(p)
    ranked = p[order]
    adjusted_ranked = ranked * len(p) / np.arange(1, len(p) + 1)
    adjusted_ranked = np.minimum.accumulate(adjusted_ranked[::-1])[::-1]
    adjusted = np.empty_like(adjusted_ranked)
    adjusted[order] = np.clip(adjusted_ranked, 0, 1)
    return adjusted


mw_rows = []
for feature in FEATURE_NAMES:
    mi = df_features.loc[df_features["label"] == "MI", feature].dropna()
    hc = df_features.loc[df_features["label"] == "HC", feature].dropna()
    if len(mi) and len(hc):
        statistic, p_value = mannwhitneyu(mi, hc, alternative="two-sided")
        mw_rows.append({"feature": feature, "MI median": mi.median(), "HC median": hc.median(),
                        "U statistic": statistic, "p-value": p_value})
df_mannwhitney = pd.DataFrame(mw_rows)
if not df_mannwhitney.empty:
    df_mannwhitney["significant p<0.05"] = df_mannwhitney["p-value"] < 0.05
    df_mannwhitney["BH adjusted p"] = benjamini_hochberg(df_mannwhitney["p-value"])
    df_mannwhitney["BH significant"] = df_mannwhitney["BH adjusted p"] < 0.05
display(df_mannwhitney.sort_values("p-value") if not df_mannwhitney.empty else df_mannwhitney)


## Q. Patient-level split — bắt buộc

Ta split danh sách bệnh nhân có stratify theo label, sau đó mới lấy các recording. Như vậy cùng một người không thể xuất hiện ở cả train và test.


In [ ]:
patient_labels = df_features[["patient_id", "label"]].drop_duplicates()
label_counts_per_patient = patient_labels.groupby("patient_id")["label"].nunique()
if (label_counts_per_patient > 1).any():
    raise ValueError("Có patient mang nhiều label MI/HC; cần kiểm tra metadata trước split")

unique_patients = patient_labels.drop_duplicates("patient_id")
if unique_patients["label"].value_counts().min() < 2:
    raise RuntimeError("Không đủ ít nhất 2 bệnh nhân mỗi lớp để split. Hãy bật RUN_FULL_DATASET.")

train_patients, test_patients = train_test_split(
    unique_patients, test_size=TEST_SIZE, random_state=RANDOM_STATE,
    stratify=unique_patients["label"]
)
train_ids, test_ids = set(train_patients["patient_id"]), set(test_patients["patient_id"])
overlap = train_ids & test_ids
assert overlap == set(), f"Patient leakage: {overlap}"

train_df = df_features[df_features["patient_id"].isin(train_ids)].copy()
test_df = df_features[df_features["patient_id"].isin(test_ids)].copy()
print("Train patients:", len(train_ids))
print("Test patients:", len(test_ids))
print("Train records:", len(train_df))
print("Test records:", len(test_df))
print("Patient overlap:", overlap)
display(pd.crosstab(train_patients["label"], columns="train patients"))
display(pd.crosstab(test_patients["label"], columns="test patients"))


## R–S. Machine Learning và metrics

- Logistic Regression và SVM: median imputation → StandardScaler → model.
- Random Forest: median imputation → model (không cần scale).
- Sensitivity = bắt đúng bao nhiêu recording MI.
- Specificity = nhận đúng bao nhiêu recording HC.

Split là patient-level nhưng metrics dưới đây là record-level trên test patients. Accuracy không được xem một mình.


In [ ]:
X_train, y_train = train_df[FEATURE_NAMES], train_df["label"]
X_test, y_test = test_df[FEATURE_NAMES], test_df["label"]

models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "SVM": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                          random_state=RANDOM_STATE, n_jobs=-1)),
    ]),
}

metric_rows, fitted_models = [], {}
for name, model in models.items():
    model.fit(X_train, y_train)
    prediction = model.predict(X_test)
    probability_mi = model.predict_proba(X_test)[:, list(model.classes_).index("MI")]
    tn, fp, fn, tp = confusion_matrix(y_test, prediction, labels=["HC", "MI"]).ravel()
    metric_rows.append({
        "model": name,
        "Accuracy": accuracy_score(y_test, prediction),
        "Sensitivity (Recall MI)": recall_score(y_test, prediction, pos_label="MI", zero_division=0),
        "Specificity": tn / (tn + fp) if (tn + fp) else np.nan,
        "F1 MI": f1_score(y_test, prediction, pos_label="MI", zero_division=0),
        "ROC-AUC": roc_auc_score((y_test == "MI").astype(int), probability_mi) if y_test.nunique() == 2 else np.nan,
    })
    fitted_models[name] = model
    ConfusionMatrixDisplay.from_predictions(y_test, prediction, labels=["HC", "MI"], cmap="Blues")
    plt.title(f"Confusion matrix — {name}")
    plt.show()

df_metrics = pd.DataFrame(metric_rows).set_index("model")
display(df_metrics)


## T. Feature importance / coefficient

Random Forest importance và Logistic coefficient chỉ phản ánh đóng góp trong mô hình/dữ liệu hiện tại; không chứng minh quan hệ nhân quả.


In [ ]:
rf = fitted_models["Random Forest"].named_steps["model"]
rf_importance = pd.Series(rf.feature_importances_, index=FEATURE_NAMES).sort_values()
plt.figure(figsize=(9, 5))
rf_importance.plot.barh()
plt.xlabel("Feature importance")
plt.title("Random Forest feature importance")
plt.show()

logistic = fitted_models["Logistic Regression"].named_steps["model"]
positive_class_index = list(logistic.classes_).index("MI")
# Binary LogisticRegression có một vector coefficient hướng về classes_[1].
direction = 1 if positive_class_index == 1 else -1
logistic_coef = pd.Series(direction * logistic.coef_[0], index=FEATURE_NAMES).sort_values()
plt.figure(figsize=(9, 5))
logistic_coef.plot.barh(color=np.where(logistic_coef >= 0, "#c44e52", "#4c72b0"))
plt.axvline(0, color="black", linewidth=1)
plt.xlabel("Coefficient sau standardization (dương → MI)")
plt.title("Logistic Regression coefficients")
plt.show()


## U. Tóm tắt toàn pipeline

1. **Dữ liệu ban đầu:** mỗi PTB record gồm header/metadata và tín hiệu 12 ECG + Frank X/Y/Z.
2. **X/Y/Z → loop:** lọc, tìm R, cắt nhiều beat, lấy median beat, rồi lấy riêng đoạn QRS và T trong không gian 3D.
3. **Beat:** một cửa sổ tín hiệu quanh một R peak.
4. **Representative beat:** median theo thời điểm của nhiều beat, đại diện cho recording.
5. **Delineation:** ước lượng `QRS_on`, `QRS_off`, `T_on`, `T_off`; phải QC và validate vì heuristic có thể sai.
6. **PCA:** tìm mặt phẳng 2D tốt nhất của từng loop để tính Arc/Area; không giảm 12 feature trước ML.
7. **12 features:** 7 QRS (Arc, Area, MaxVec, MaxCentroidDistance, Max/Mean/StdVel) và 5 T (MaxVec, Area, Max/Mean/StdVel).
8. **Dataset ML:** một hàng/recording, gồm `patient_id`, `record_id`, đúng 12 feature và `label`.
9. **Model:** input chỉ là 12 VCG features; output là MI hoặc HC. Diagnosis chỉ tạo label; ID chỉ quản lý/truy vết và split.

```text
PTB
↓
patient + diagnosis + X/Y/Z
↓
filter
↓
R peaks
↓
beats
↓
representative beat
↓
QRS/T boundaries
↓
QRS/T loops
↓
PCA
↓
12 features
↓
statistical analysis
↓
patient-level ML
↓
MI / HC
```

Trước khi dùng cho bài báo/luận văn: validate Savitzky–Golay interpretation và QRS/T delineation với tài liệu tham chiếu, xem failure/QC table, và cân nhắc patient-level/repeated-measures inference.
